In [19]:
import numpy as np
import pandas as pd
from plotly import graph_objects as go
import nevergrad as ng
from summer2 import AgeStratification, Overwrite

from estival.wrappers.nevergrad import optimize_model
import estival.priors as esp
import estival.targets as est
from estival.model import BayesianCompartmentalModel
from estival.wrappers import pymc as epm
pd.options.plotting.backend = "plotly"
from summer2 import CompartmentalModel
from summer2.parameters import Parameter, DerivedOutput
from summer2.functions.time import get_sigmoidal_interpolation_function

from tb_incubator.demographics import add_extra_crude_birth_flow
from tb_incubator.constants import set_project_base_path

project_paths = set_project_base_path("../incubator2024/tb_incubator")

In [3]:
def get_age_groups_in_range(age_groups, lower_limit, upper_limit):
    return [i for i in age_groups if '+' not in i and lower_limit <= int(i.split('-')[0]) <= upper_limit]

In [4]:
# UN birth data
data_path = project_paths["DATA_PATH"]
birth_rates = pd.read_csv(data_path / 'id_birth.csv', index_col=0)['value']

In [5]:
# population & deaths
un_pop_data = pd.read_csv(data_path / "id_pop_deaths.csv")
un_pop_data

,year,age,population,deaths
0,1950,0-4,10399052,814
1,1950,10-14,8729789,41
2,1950,100+,30,0
3,1950,15-19,7742333,46
4,1950,20-24,6603299,54
...,...,...,...,...
1549,2023,75-79,3387201,250
1550,2023,80-84,1927361,228
1551,2023,85-89,777528,143
1552,2023,90-94,185325,51


In [6]:
un_pops = pd.DataFrame({
    'population': un_pop_data["population"],
    'deaths': un_pop_data["deaths"]
})
un_pops.index = pd.MultiIndex.from_frame(un_pop_data[["year", "age"]])

target_pops = un_pops.groupby(level=[0]).sum()["population"]
target_deaths = un_pops.groupby(level=[0]).sum()["deaths"]

In [7]:
age_groups = set(un_pops.index.get_level_values(1))
years = set(un_pops.index.get_level_values(0))

In [8]:
un_pops

population  deaths
year age                      
1950 0-4      10399052     814
     10-14     8729789      41
     100+           30       0
     15-19     7742333      46
     20-24     6603299      54
...                ...     ...
2023 75-79     3387201     250
     80-84     1927361     228
     85-89      777528     143
     90-94      185325      51
     95-99       23068      10

[1554 rows x 2 columns]

In [9]:
agegroup_request = [[0, 4], [5, 14], [15, 34], [35, 49], [50, 74]]
agegroup_map = {low: get_age_groups_in_range(age_groups, low, up) for low, up in agegroup_request}
agegroup_map[agegroup_request[-1][0]].append('75+')
agegroup_map

{0: ['0-4'],
 5: ['10-14', '5-9'],
 15: ['15-19', '25-29', '30-34', '20-24'],
 35: ['35-39', '45-49', '40-44'],
 50: ['65-69', '70-74', '55-59', '50-54', '60-64', '75+']}

In [10]:
mapped_rates = pd.DataFrame()
for year in years:
    for agegroup in agegroup_map:
        age_mask = [i in agegroup_map[agegroup] for i in un_pops.index.get_level_values(1)]
        age_year_data = un_pops.loc[age_mask].loc[year, :]
        total = age_year_data.sum()
        mapped_rates.loc[year, agegroup] = total['deaths'] / total['population']
death_rates = mapped_rates.loc[birth_rates.index]

In [11]:
birth_rates

year
1950    0.040454
1951    0.041152
1952    0.041820
1953    0.042415
1954    0.043055
          ...   
2019    0.016909
2020    0.016674
2021    0.016417
2022    0.016204
2023    0.015941
Name: value, Length: 74, dtype: float64

In [12]:
# Base model construction - also quite arbitrary
model_comps = ["susceptible", "early latent", "late latent", "infectious", "recovered"]
model_times = [1850.0, 2024.0]
model = CompartmentalModel(
    times=model_times,
    compartments=model_comps,
    infectious_compartments=["infectious"],
)
init_pops = {"susceptible": Parameter("starting population"), "infectious": 0.0}
model.set_initial_population(init_pops)


In [22]:
# TB transitions, some meaningless TB-related flows
model.add_death_flow("TB death", Parameter("death rate"), "infectious")


In [14]:
# Demographic transitions
model.add_universal_death_flows("population_death", Parameter("population death rate"))
model.add_replacement_birth_flow("replacement_birth", "susceptible")
add_extra_crude_birth_flow(model, "extra_birth", Parameter("population growth rate"), "susceptible")

In [15]:
params = {
    "death rate": 0.1,
    "population death rate": 0.01,
}

In [23]:
age_strata = [0,5,15,35,50]
strat = AgeStratification("age", age_strata, model_comps)
universal_death_funcs, death_adjs = {}, {}
for age in age_strata:
    universal_death_funcs[age] = get_sigmoidal_interpolation_function(
        death_rates.index, death_rates[age]
    )
    death_adjs[str(age)] = Overwrite(universal_death_funcs[age])
strat.set_flow_adjustments("population_death", death_adjs)
model.stratify_with(strat)

In [ ]:
model.request_output_for_compartments("total_population", model_comps);

In [ ]:
# Prepare calibration model
priors = [
    esp.UniformPrior("population growth rate", (0.005, 0.03)),
    esp.UniformPrior("starting population", (1e6, 3e7)),
]
targets = [est.NegativeBinomialTarget("total_population", target_pops, dispersion_param=100.0)]
bcm = BayesianCompartmentalModel(model, params, priors, targets)

In [ ]:
# Set up optimisation
budget = 1000
opt_class = ng.optimizers.NGOpt
orunner = optimize_model(bcm, opt_class=opt_class, budget=budget)
start_params = {"population growth rate": 0.01, "starting population": 1.5e6}
orunner = optimize_model(bcm, opt_class=opt_class, suggested=start_params, init_method="midpoint")

In [ ]:
rec = orunner.minimize(budget)
map_params = rec.value[1]
print("Best candidate parameters:")
for i_param, param in enumerate(map_params):
    print(f"   {param}: {round(map_params[param], 4)} (within bound {priors[i_param].bounds()}")

In [ ]:
model.run(parameters=params | map_params)

In [ ]:
outputs = model.get_derived_outputs_df()
fig = go.Figure()
fig.add_trace(go.Scatter(x=outputs.index, y=outputs["total_population"], name="modelled"))
fig.add_trace(go.Scatter(x=target_pops.index, y=target_pops, name="target", line={"width": 0.0}))
fig.update_layout(yaxis={"range": [0.0, 3e8]})